# UTS Data Science (IF404) - Project-Based Learning
## Analisis Harga dan Kategori Buku Impor pada E-Commerce Periplus.com

**Dosen Pengampu:** Ir. Ahmad Chusyairi, M.Com., CDS., IPM., ASEAN Eng  
**Nama Mahasiswa:** Gading Nasution  
**Program Studi:** PJJ Informatika S1  

---

## 1. Import Library & Load Dataset

Langkah pertama adalah memuat library Python yang diperlukan untuk analisis data, manipulasi data, dan visualisasi data. Kita memuat dataset bersih `data/periplus_books_clean.csv` yang telah dikumpulkan melalui proses scraping dari situs Periplus.com.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style visualisasi agar rapi dan seragam
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset bersih
df = pd.read_csv("data/periplus_books_clean.csv")
df.head()

## 2. Eksplorasi Data & Pengecekan Kualitas Data

Sebelum melakukan analisis statistik lebih lanjut, kita perlu melakukan inspeksi terhadap kualitas data, meliputi tipe data, jumlah entri, dan pengecekan apakah terdapat data yang kosong (*missing values*).

In [ ]:
# Cek informasi dasar dataset
print("=== Informasi Dataset ===")
df.info()

# Cek apakah ada missing values
print("\n=== Jumlah Missing Values per Kolom ===")
print(df.isnull().sum())

## 3. Statistika Deskriptif & Peringkasan Data

Kita menghitung nilai statistik deskriptif untuk variabel numerik `price_idr` (harga jual buku dalam Rupiah). Ini mencakup:
- **Ukuran Pemusatan Data:** Rata-rata (Mean), Nilai Tengah (Median), dan Modus (Mode).
- **Ukuran Penyebaran Data:** Standar Deviasi, Varians, Rentang (Min-Max), Kuartil (Q1 & Q3), dan Interquartile Range (IQR).
- **Deteksi Pencilan (Outliers):** Menggunakan metode batas $Q1 - 1.5 \times IQR$ dan $Q3 + 1.5 \times IQR$.

In [ ]:
prices = df['price_idr']

# Ukuran Pemusatan
mean_val = prices.mean()
median_val = prices.median()
mode_val = prices.mode()[0] if not prices.mode().empty else None

# Ukuran Penyebaran
std_val = prices.std()
var_val = prices.var()
min_val = prices.min()
max_val = prices.max()
range_val = max_val - min_val

# Kuartil & IQR
q1 = prices.quantile(0.25)
q3 = prices.quantile(0.75)
iqr = q3 - q1

# Deteksi Outliers
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = df[(prices < lower_bound) | (prices > upper_bound)]

print(f"Mean (Rata-rata)     : Rp {mean_val:,.2f}")
print(f"Median (Nilai Tengah): Rp {median_val:,.2f}")
print(f"Mode (Modus)         : Rp {mode_val:,.2f}")
print(f"Standar Deviasi      : Rp {std_val:,.2f}")
print(f"Varians              : Rp {var_val:,.2f}")
print(f"Rentang (Min - Max)  : Rp {min_val:,.2f} - Rp {max_val:,.2f} (Range: Rp {range_val:,.2f})")
print(f"Kuartil 1 (Q1)       : Rp {q1:,.2f}")
print(f"Kuartil 3 (Q3)       : Rp {q3:,.2f}")
print(f"IQR                  : Rp {iqr:,.2f}")
print(f"Batas Bawah Outlier  : Rp {lower_bound:,.2f}")
print(f"Batas Atas Outlier   : Rp {upper_bound:,.2f}")
print(f"Jumlah Outliers      : {len(outliers)} produk")

## 4. Statistika Deskriptif per Kategori

Untuk memahami sebaran harga antar kategori buku impor, kita mengelompokkan data berdasarkan kolom `category` dan menghitung metrik statistik utamanya.

In [ ]:
category_stats = df.groupby('category')['price_idr'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
category_stats.columns = ['Jumlah Buku', 'Mean (Rata-rata)', 'Median', 'Std Deviasi', 'Min Price', 'Max Price']
category_stats

## 5. Visualisasi Data

Visualisasi membantu kita melihat pola data secara intuitif. Di sini kita memvisualisasikan:
1. **Histogram:** Untuk melihat bentuk distribusi frekuensi harga buku impor.
2. **Boxplot:** Untuk mengidentifikasi pencilan (outliers) harga buku secara keseluruhan.
3. **Bar Chart:** Untuk melihat proporsi jumlah sampel buku per kategori.
4. **Boxplot per Kategori:** Untuk membandingkan sebaran harga dan mendeteksi outlier di masing-masing kategori.

In [ ]:
# 1. Histogram Distribusi Harga
plt.figure(figsize=(10, 5))
sns.histplot(df['price_idr'], kde=True, color='skyblue', bins=30)
plt.title('Distribusi Harga Buku Impor di Periplus.com', fontsize=14, fontweight='bold')
plt.xlabel('Harga (Rupiah)', fontsize=12)
plt.ylabel('Frekuensi', fontsize=12)
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))
plt.tight_layout()
plt.show()

# 2. Boxplot Harga Buku (Deteksi Outlier)
plt.figure(figsize=(10, 3))
sns.boxplot(x=df['price_idr'], color='lightcoral')
plt.title('Boxplot Harga Buku Impor (Deteksi Outlier)', fontsize=14, fontweight='bold')
plt.xlabel('Harga (Rupiah)', fontsize=12)
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))
plt.tight_layout()
plt.show()

# 3. Bar Chart Jumlah Buku per Kategori
plt.figure(figsize=(10, 5))
category_counts = df['category'].value_counts()
sns.barplot(x=category_counts.values, y=category_counts.index, hue=category_counts.index, palette='viridis', legend=False)
plt.title('Jumlah Buku per Kategori dalam Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Buku', fontsize=12)
plt.ylabel('Kategori', fontsize=12)
plt.tight_layout()
plt.show()

# 4. Boxplot Harga Buku per Kategori
plt.figure(figsize=(12, 6))
sns.boxplot(x='price_idr', y='category', data=df, hue='category', palette='Set2', legend=False)
plt.title('Perbandingan Sebaran Harga Buku per Kategori', fontsize=14, fontweight='bold')
plt.xlabel('Harga (Rupiah)', fontsize=12)
plt.ylabel('Kategori', fontsize=12)
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))
plt.tight_layout()
plt.show()

## 6. Temuan Utama & Rekomendasi Bisnis

### Temuan Utama:
1. **Distribusi Condong Kanan (*Right-Skewed*):** Rata-rata harga buku (Rp 338,934.06) berada di atas nilai median (Rp 285,000.00). Hal ini disebabkan oleh adanya buku-buku dengan harga sangat tinggi (ekstrim kanan).
2. **Kategori Termahal:** Kategori **Computer & IT** memiliki rata-rata harga tertinggi (Rp 527,757.92) dan variasi harga terluas (mencapai Rp 1,951,000.00). Ini mencerminkan tingginya biaya buku referensi akademis dan profesional IT.
3. **Kategori Termurah:** Kategori **Children's Books** memiliki rata-rata harga terendah (Rp 229,533.75) dengan sebaran harga yang paling rapat/konsisten.
4. **Outliers:** Terdeteksi sebanyak 37 produk buku impor dengan harga di atas Rp 672,500.00, sebagian besar berasal dari kategori Computer & IT dan Biographies & Memoirs.

### Rekomendasi Bisnis untuk Periplus.com:
1. **Strategi Bundling Buku Anak:** Karena harga *Children's Books* cenderung murah dan stabil, Periplus dapat membuat paket *bundling* berisi 3-4 buku anak dengan harga diskon paket guna meningkatkan nilai transaksi rata-rata (*Average Order Value*).
2. **Program Cicilan/Promo untuk Buku IT:** Mengingat harga buku *Computer & IT* sangat tinggi (Outliers), Periplus dapat bekerja sama dengan penyedia pembayaran (*Paylater* atau cicilan bank) untuk mempermudah pembelian buku-buku IT profesional.
3. **Katalog Khusus Buku di Bawah Rp 250,000:** Karena median harga buku fiksi dan biografi berada di kisaran Rp 250k - Rp 280k, membuat promo *landing page* khusus "Buku Impor di Bawah Rp 250,000" akan sangat menarik minat pembeli impulsif.